# Water use processing

Create a single netcdf with all variables

In [ ]:
# basic
from pathlib import Path
import yaml
import numpy as np
import os

# spatial
import xarray as xr

from src.utils import load_config
cfg = load_config()
    path_data_raw = Path(cfg["dir"]["data"])

In [ ]:
common_period = slice("1990-01-01", "2010-12-31")

## Data

In [ ]:
# local data
ds_cr2wu_nlu = xr.open_dataset(path_data_raw / "WATER_USE/CR2WU/CR2WU_NLU_1960_2020.nc").Total_NLU
ds_cr2wu_nlu = ds_cr2wu_nlu.sel(time = common_period)
ds_cr2wu_nlu = ds_cr2wu_nlu.mean(dim = "time")

ds_cr2wu_lulucf = xr.open_dataset(path_data_raw / "WATER_USE/CR2WU/CR2WU_LULUCF_1960_2020.nc")[["Irrigated_Consumptive", "Rainfed_Consumptive"]]
ds_cr2wu_lulucf = ds_cr2wu_lulucf.Irrigated_Consumptive + ds_cr2wu_lulucf.Rainfed_Consumptive
ds_cr2wu_lulucf = ds_cr2wu_lulucf.sel(time = common_period)
ds_cr2wu_lulucf = ds_cr2wu_lulucf.mean(dim = "time")
ds_cr2wu = ds_cr2wu_nlu + ds_cr2wu_lulucf

# global data
ds_global  = xr.open_dataset(path_data_raw / "WATER_USE/Huang_HESS/Water_use_Huang.nc").cons_total
ds_global  = ds_global.sel(date=common_period)
ds_global  = ds_global.resample(date="YS").sum()
ds_global  = ds_global.mean(dim="date")

In [ ]:
# Extend CR2WU grid to the full study area before interpolating Huang
dx = float(np.abs(ds_cr2wu.x.diff("x").median()))
dy = float(np.abs(ds_cr2wu.y.diff("y").median()))

x_min_target = float(ds_global.lon.min())
x_max_target = float(ds_global.lon.max())
y_min_target = float(ds_global.lat.min())
y_max_target = float(ds_global.lat.max())

x_min0 = float(ds_cr2wu.x.min())
x_max0 = float(ds_cr2wu.x.max())
y_min0 = float(ds_cr2wu.y.min())
y_max0 = float(ds_cr2wu.y.max())

n_west  = max(0, int(np.ceil((x_min0 - x_min_target) / dx)))
n_east  = max(0, int(np.ceil((x_max_target - x_max0) / dx)))
n_north = max(0, int(np.ceil((y_max_target - y_max0) / dy)))
n_south = max(0, int(np.ceil((y_min0 - y_min_target) / dy)))

x_west = x_min0 - dx * np.arange(n_west, 0, -1)
x_east = x_max0 + dx * np.arange(1, n_east + 1)
y_north = y_max0 + dy * np.arange(n_north, 0, -1)
y_south = y_min0 - dy * np.arange(1, n_south + 1)

new_x = np.concatenate([x_west, ds_cr2wu.x.values, x_east])
new_y = np.concatenate([y_north, ds_cr2wu.y.values, y_south])

ds_cr2wu = ds_cr2wu.reindex(x=new_x, y=new_y)

ds_global_interp = ds_global.interp(
    lon=ds_cr2wu.x.values,
    lat=ds_cr2wu.y.values,
    method="nearest",
)

ratio = ds_cr2wu.rename({"y": "lat", "x": "lon"}) / ds_global_interp
ratio = ratio.where(np.isfinite(ratio) & (ratio > 0))

# Load full Huang time series and apply correction factor
ds_global_full = xr.open_dataset(path_data_raw / "WATER_USE/Huang_HESS/Water_use_Huang.nc").cons_total
ds_global_full = ds_global_full.interp(lon=ds_cr2wu.x.values, lat=ds_cr2wu.y.values, method="nearest")
ds_global_corrected = (ds_global_full * ratio.fillna(1))
ds_global_corrected = ds_global_corrected.isel(lat=slice(None, None, -1))

ds_global_corrected = ds_global_corrected.rename("cons_total").to_dataset()
ds_global_corrected.to_netcdf(path_data_raw / "WATER_USE/Huang_HESS/Water_use_Huang_corrected.nc")